# Session 2 demo: two ways to give a model tools

One small agent, built with the OpenAI Agents SDK. The demo runs in two parts:

- **Part 1, after the tool calling slides.** A local tool: a Python function in this notebook that looks things up in a small piece of JSON data. Then its trace in Arize.
- **Part 2, after the MCP slides.** A tool from an MCP server: DeepWiki's public server, which answers questions about any GitHub repository. Someone else wrote it and runs it, and it needs no sign-in. Then both tools in one run, and that trace in Arize.

Either way, the model sees a tool definition and gets a result back.

## Setup

Connect to a standard Python runtime in Colab. Run the install cell, then the configuration cell below. The demo data and tool are included in this notebook.

In [ ]:
# @title Install dependencies { display-mode: "form" }

import sys

if "google.colab" in sys.modules:
    %pip install -q uv

# Use the workshop pins and keep Colab's bundled Google ADK compatible.
!uv pip install -q openai==2.31.0 openai-agents==0.13.6 arize==8.64.0 arize-otel==0.12.0 openinference-instrumentation-openai-agents==1.4.1 pandas==2.3.3 python-dotenv==1.2.2 ipywidgets==8.1.9 opentelemetry-exporter-otlp==1.42.1

<details>
<summary>Configure Colab Secrets</summary>

Open **Secrets** (the key icon in the left sidebar). Add or reuse these entries and enable **Notebook access** for each one:

- **Arize traces:** `ARIZE_API_KEY` and `ARIZE_SPACE_ID`.
- **Egen Azure OpenAI (default):** `AZURE_OPENAI_API_KEY`. The training proxy URL is already set below. Set `MODEL` to your Azure deployment name if it differs from the default.
- **Direct OpenAI (alternative):** `OPENAI_API_KEY`. Set `PROVIDER = "openai"` and choose a Responses API model available to your account.

Run the next cell, then work through Part 1 and Part 2 in order.

</details>

In [ ]:
# @title Configure the model and Arize { display-mode: "form" }

PROVIDER = "azure"       # "azure" or "openai"
MODEL = "gpt-5.6-terra"  # Azure deployment name. For direct OpenAI, e.g. "gpt-6-luna".

AZURE_OPENAI_ENDPOINT = "https://oaitrain-proxy.wittyisland-3158b851.centralus.azurecontainerapps.io/v1"

import json
import os
from datetime import datetime

from agents import Agent, OpenAIResponsesModel, RunConfig, Runner, function_tool
from arize.otel import register
from openai import AsyncOpenAI
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def read_secret(name):
    if userdata is None:
        return os.environ.get(name)
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


provider_keys = {
    "azure": ("AZURE_OPENAI_API_KEY",),
    "openai": ("OPENAI_API_KEY",),
}
if PROVIDER not in provider_keys:
    raise ValueError('Set PROVIDER to "azure" or "openai".')

required_keys = (*provider_keys[PROVIDER], "ARIZE_API_KEY", "ARIZE_SPACE_ID")
secrets = {name: read_secret(name) for name in required_keys}
missing = [name for name, value in secrets.items() if not value]
if missing:
    raise RuntimeError(
        f"Missing or inaccessible secrets: {', '.join(missing)}. "
        "Check the names and enable Notebook access in Colab's Secrets panel."
    )

if PROVIDER == "azure":
    client = AsyncOpenAI(
        base_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
        api_key=secrets["AZURE_OPENAI_API_KEY"],
    )
else:
    client = AsyncOpenAI(api_key=secrets["OPENAI_API_KEY"])
model = OpenAIResponsesModel(MODEL, client)

project_name = f"Egen-Tools-MCP-Demo-{datetime.now():%Y-%m-%d}"
instrumentor = OpenAIAgentsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()
tracer_provider = register(
    space_id=secrets["ARIZE_SPACE_ID"],
    api_key=secrets["ARIZE_API_KEY"],
    project_name=project_name,
    set_global_tracer_provider=False,
    verbose=False,
)
instrumentor.instrument(tracer_provider=tracer_provider)

print(f"Configured {PROVIDER}: {MODEL}. Arize project: {project_name}")


def named(trace_name):
    """Give a run a name, so its trace is easy to find in Arize."""
    return RunConfig(workflow_name=trace_name)


def show(result, local=()):
    """Print each tool the model asked for, what came back, and the final answer."""
    calls = {}
    for item in result.new_items:
        if item.type == "tool_call_item":
            calls[item.raw_item.call_id] = [item.raw_item, None]
        elif item.type == "tool_call_output_item":
            calls[item.raw_item["call_id"]][1] = item.output
    for step, (call, output) in enumerate(calls.values(), 1):
        args = ", ".join(f"{k}={v!r}" for k, v in json.loads(call.arguments or "{}").items())
        where = "local Python function" if call.name in local else "MCP server"
        output = output.get("text", output) if isinstance(output, dict) else output
        text = " ".join(str(output).split())
        print(f"{step}. The model asks for  {call.name}({args})   [{where}]")
        print(f"   It gets back        {text[:300]}{' ...' if len(text) > 300 else ''}")
    print("\nAnswer:", result.final_output)

---
# Part 1: a local tool

*Run this part after the tool calling slides.*

A tool is a function in our own code, plus a description the model can read. The model never runs it. It asks for a call, and our code runs the function and sends the result back.

### The data

A few made-up client projects and the open-source tools each one runs on. The model has never seen it.

In [ ]:
# @title Load project records { display-mode: "form" }

PROJECTS = json.loads('''
[
  {"project": "Atlas",   "goal": "Nightly sales reporting for a grocery chain",
   "stack": [{"tool": "Airflow", "repo": "apache/airflow"}, {"tool": "dbt", "repo": "dbt-labs/dbt-core"}]},
  {"project": "Beacon",  "goal": "Real-time fraud alerts for a bank",
   "stack": [{"tool": "Kafka", "repo": "apache/kafka"}, {"tool": "Flink", "repo": "apache/flink"}]},
  {"project": "Compass", "goal": "A support assistant for an insurer",
   "stack": [{"tool": "LangGraph", "repo": "langchain-ai/langgraph"}, {"tool": "FastAPI", "repo": "fastapi/fastapi"}]}
]
''')

In [ ]:
# @title Show project data { display-mode: "form" }

for project in PROJECTS:
    stack = ", ".join(f"{item['tool']} ({item['repo']})" for item in project["stack"])
    print(f"{project['project']} — {project['goal']} | {stack}")

### The tool

One function. The docstring is the description the model reads.

In [ ]:
@function_tool
def get_project(name: str) -> dict:
    '''Look up one of our client projects by name. Returns its goal and the open-source tools it runs on, each with its GitHub repository.'''
    return next((p for p in PROJECTS if p["project"].lower() == name.lower()), {"error": f"No project called {name}"})


LOCAL = {"get_project"}

# This is everything the model is told about the tool: a name, a description and the arguments.
print(json.dumps({"name": get_project.name, "description": get_project.description, "parameters": get_project.params_json_schema}, indent=2))

### Ask a question that needs it

In [ ]:
INSTRUCTIONS = "You help engineers at a consulting firm. Be brief. For a project’s stack, call get_project. For questions about what an open-source tool does, look it up with DeepWiki."

agent = Agent(name="Demo assistant", instructions=INSTRUCTIONS, model=model, tools=[get_project])
result = await Runner.run(agent, "What does the Atlas project run on?", run_config=named("Part 1: local tool"))
show(result, LOCAL)

### When the tool has no answer

The function returns an error the model can read, and the model passes that on.

In [ ]:
# @title Try an unknown project { display-mode: "form" }

result = await Runner.run(agent, "What does the Zephyr project run on?", run_config=named("Part 1: project not found"))
show(result, LOCAL)

<details>
<summary>See it in Arize</summary>

Open [Arize](https://app.arize.com) → **Projects** → the project printed in setup, and open the trace named **Part 1: local tool**.

- Look for two model calls with `get_project` between them.
- Click the **first model call**. Its input holds one tool definition, the same JSON printed above. Its output is not text: it is a request to call `get_project` with `name = "Atlas"`.
- Click the **`get_project` span**. That is our Python function running: arguments in, the project out.
- Click the **second model call**. The tool's result is now part of its input, and its output is the answer.

</details>

---
**Stop here.** Back to the slides for MCP.

---
# Part 2: a tool from an MCP server

*Run this part after the MCP slides.*

Now the tools come from a server we did not write. We connect, ask the server which tools it has, and hand those definitions to the model.

[DeepWiki](https://deepwiki.com) answers questions about public GitHub repositories. Its MCP server needs no sign-in.

### Connect and discover

In [ ]:
from agents.mcp import MCPServerStreamableHttp

DEEPWIKI = {"url": "https://mcp.deepwiki.com/mcp", "timeout": 30}


def deepwiki():
    return MCPServerStreamableHttp(name="deepwiki", params=DEEPWIKI, client_session_timeout_seconds=120, cache_tools_list=True)


# Discovery: the client asks the server what it offers.
async with deepwiki() as server:
    for tool in await server.list_tools():
        arguments = ", ".join(tool.inputSchema.get("properties", {}))
        print(f"{tool.name}({arguments})\n    {tool.description.strip().splitlines()[0]}\n")

### Ask a question that needs the server

In [ ]:
async with deepwiki() as server:
    agent = Agent(name="Demo assistant", instructions=INSTRUCTIONS, model=model, mcp_servers=[server])
    result = await Runner.run(agent, "In two sentences, what is dbt? Its repository is dbt-labs/dbt-core.", run_config=named("Part 2: MCP server"))
show(result, LOCAL)

### Both tools in one run

The same agent with both kinds of tool. Which tool is the "other" one is only in our data, and what that tool does comes from DeepWiki. The model does not know or care where each tool lives.

In [ ]:
async with deepwiki() as server:
    agent = Agent(name="Demo assistant", instructions=INSTRUCTIONS, model=model, tools=[get_project], mcp_servers=[server])
    result = await Runner.run(agent, "I'm joining the Atlas project next week. What does it run on? I know Airflow, so give me a one-line primer on the other tool.", run_config=named("Part 2: both tools"))
show(result, LOCAL)

<details>
<summary>See it in Arize</summary>

In the same project, open the trace named **Part 2: both tools**.

- Look for three model calls, with `get_project` and then a DeepWiki tool call between them.
- Click any model call. Its input now holds our Python function and the tools DeepWiki advertised in the discovery cell. To the model they look the same.
- Click the **DeepWiki tool span**. The repository in the DeepWiki call, `dbt-labs/dbt-core`, came out of our JSON in the step before.
- Compare the two tool spans. `get_project` runs in the notebook. Compare its duration with the call to DeepWiki’s server.

</details>

---
## Optional: a different server is one URL away

Any MCP server works the same way. OpenAI's developer docs server also needs no sign-in.

In [ ]:
# @title List OpenAI docs MCP tools { display-mode: "form" }

async with MCPServerStreamableHttp(name="openai-docs", params={"url": "https://developers.openai.com/mcp", "timeout": 30}) as server:
    for tool in await server.list_tools():
        print(f"{tool.name}: {tool.description.strip().splitlines()[0][:100]}")